## Token & Param

In [1]:
vocab_size = 50000
dhead = 64
k = 8
dmodel = dhead * k
print(f'dmodel: {dmodel}')
n_blocks = k

enc_params = n_blocks * 12 * dmodel ** 2
emb_params = dmodel * vocab_size
params = enc_params + emb_params
print(f'params: {params / 1e6:.0f}M\t encoder: {enc_params/1e6:.0f}M\tembedding: {emb_params/1e6:.0f}M')

dmodel: 512
params: 51M	 encoder: 25M	embedding: 26M


In [2]:
tokens = None
# tokens = 125e3 * 512 * 512
if tokens is None:
    tokens = 40 * params

ratio = tokens / params
print(f'ratio: {ratio}')
print(f'tokens: {tokens / 1e9:.2f}B')

ratio: 40.0
tokens: 2.03B


In [3]:
batch_size = 512
seq_len = 512
n_steps = tokens / (batch_size * seq_len)
print(f'n steps: {n_steps:.0f}')

n steps: 7746


In [18]:
float_bytes = 4
tokens_memory = float_bytes * dmodel * batch_size * seq_len
print(f'batch_memory: {tokens_memory / 1e6:.0f} MB')

model_memory = 4 * float_bytes * params
print(f'model_memory: {model_memory / 1e9:.0f} GB')

batch_memory: 1879 MB
model_memory: 19 GB


## compute cost $$$

In [19]:
mfu = 0.3
gpu_flops = (1671 / 2) * 1e12

In [20]:
theoretical_flops = 6 * tokens * params
print(f'theoretical_flops: {theoretical_flops / 1e18:.2f}*1E6 TFLOPS')
gpu_hours = theoretical_flops / (gpu_flops * 3600 * mfu)
print(f'GPU hours: {gpu_hours:.1f}')
n_gpus = 4
print(f'training hours: {gpu_hours / n_gpus:.1f}')
print(f'training days: {gpu_hours / (n_gpus * 24):.1f}')

theoretical_flops: 327.74*1E6 TFLOPS
GPU hours: 363.2
training hours: 90.8
training days: 3.8


In [40]:
grid_size = 15
total_gpu_hours = grid_size * gpu_hours
print(f'total_gpu_hours: {total_gpu_hours:.0f}')

total_gpu_hours: 6302


## Calculate MFU

In [8]:
minutes = 60 * 13.5
steps = 50e3
batch_size = 512
seq_len = 512
vocab_size = 50000
k = 24
gpu_flops = (1671 / 2) * 1e12
n_gpus = 4
dhead = 64
dmodel = dhead * k
print(f'dmodel: {dmodel}')
n_blocks = k

params = n_blocks * 12 * dmodel ** 2 + dmodel * vocab_size
print(f'params: {params / 1e6:.0f}M')

dmodel: 1536
params: 756M


In [9]:
tokens_processed = steps * batch_size * seq_len
th_flops = 6 * tokens_processed * params
real_flops = gpu_flops * n_gpus * 60 * minutes
mfu = th_flops / real_flops
print(f'MFU: {mfu:.3f}')

MFU: 0.366
